# Story 11: Couple Python to Simulation Codes which Use MPI Internally
**Who:** A computational scientist whose physics lives in an established simulation code written in C, C++ or
Fortran and parallelized with MPI - [Quantum Espresso](https://www.quantum-espresso.org),
[LAMMPS](https://www.lammps.org), VASP, CP2K or an in-house code. The workflow around it - generating input files,
scanning parameters, analysing the outputs - is written in Python.

**What they want:** Run many simulations from one Python script, each with several MPI ranks in its own directory, and
get the parsed results back as Python objects.

**Why plain SLURM is painful:** For every simulation a directory is created, an input file written, a batch script
with an `srun` line generated and submitted. Afterwards a second script walks over the directories, checks which runs
finished and parses the outputs. The logic of the workflow is spread over Python, shell scripts and files. mpi4py does
not help either: the simulation code is a separate executable which cannot use a communicator created in Python.

**How executorlib helps:** The simulation is wrapped in an ordinary Python function which writes the input file,
starts the executable with the MPI launcher and parses the output file. The `SlurmClusterExecutor` submits every call
of this function as a SLURM job with its own working directory and enough CPU cores for the MPI ranks of the
executable, and returns the parsed results as futures.

This notebook runs on the **login node** of a SLURM cluster.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

`mpi_launcher` starts the MPI executable inside each job, for example `["srun"]` with the MPI plugin of your cluster or
`["mpiexec"]`. `mpicc` compiles the small example code of Step 1 - on your cluster load the compiler and MPI modules
and use `["mpicc"]`. On the test cluster the MPICH compiler wrapper of the conda environment is used with the system C
compiler (`-cc=gcc`), and the location of the MPI library is embedded in the executable (`-Wl,-rpath,...`), so the
executable uses the same MPI library as mpi4py. `cores_per_simulation` is the number of MPI ranks of every simulation.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
import os
import sys

submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env=L
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""
mpi_launcher = ["srun", "--mpi=pmi2"]  # starts the MPI executable inside the job, e.g. ["mpiexec"]
mpicc = ["mpicc", "-cc=gcc", "-Wl,-rpath," + os.path.join(sys.prefix, "lib")]  # MPI C compiler, e.g. ["mpicc"]
cores_per_simulation = 2  # MPI ranks of each simulation

## Step 1: The simulation code
In a real project the simulation code is already installed, for example with `module load quantum-espresso`. To keep
this notebook self-contained, it writes and compiles a minimal MPI program in C instead. Like a real simulation code,
it reads an input file from the current directory, distributes the work over its MPI ranks, combines the partial
results with `MPI_Reduce` and writes an output file: it computes pi by integrating `4 / (1 + x^2)` from 0 to 1 with
`n_intervals` intervals.

In [ ]:
c_source = r"""
#include <mpi.h>
#include <stdio.h>

/* Minimal MPI "simulation code": integrate 4 / (1 + x^2) from 0 to 1, which equals pi. */
int main(int argc, char **argv) {
    int rank, size;
    long n_intervals = 0;
    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    if (rank == 0) {  /* read the input file */
        FILE *input = fopen("input.txt", "r");
        if (input == NULL || fscanf(input, "n_intervals %ld", &n_intervals) != 1 || n_intervals < 1) {
            fprintf(stderr, "invalid input.txt\n");
            MPI_Abort(MPI_COMM_WORLD, 1);
        }
        fclose(input);
    }
    MPI_Bcast(&n_intervals, 1, MPI_LONG, 0, MPI_COMM_WORLD);
    double width = 1.0 / n_intervals, local_sum = 0.0, pi = 0.0;
    for (long i = rank; i < n_intervals; i += size) {  /* every rank handles every size-th interval */
        double x = width * (i + 0.5);
        local_sum += 4.0 / (1.0 + x * x);
    }
    local_sum *= width;
    MPI_Reduce(&local_sum, &pi, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
    if (rank == 0) {  /* write the output file */
        FILE *output = fopen("output.txt", "w");
        fprintf(output, "pi %.15f\nmpi_ranks %d\n", pi, size);
        fclose(output);
        printf("computed pi with %ld intervals on %d MPI ranks\n", n_intervals, size);
    }
    MPI_Finalize();
    return 0;
}
"""

The code is compiled once on the login node. On clusters whose login and compute nodes have different CPU
architectures, compile it in a job instead - or use the installed module. The jobs run in other directories, so the
absolute path of the executable is used.

In [ ]:
import subprocess

code_directory = os.path.abspath("story11_code")
os.makedirs(code_directory, exist_ok=True)
source_file = os.path.join(code_directory, "pi_mpi.c")
executable = os.path.join(code_directory, "pi_mpi")

with open(source_file, "w") as f:
    f.write(c_source)
subprocess.run(mpicc + ["-O2", "-o", executable, source_file], check=True)

## Step 2: Wrap the simulation in a Python function
The function covers one simulation from input to parsed result: it writes the input file, starts the executable with
`mpi_ranks` MPI ranks and parses the output file. The standard output of the simulation is written to a log file, and
`check=True` raises an exception if the simulation fails, which executorlib passes on to the future. The function
works in the current directory - executorlib sets it per call with `cwd` in Step 3.

[ASE](https://wiki.fysik.dtu.dk/ase) provides such wrappers for many simulation codes as calculators: the
[Quantum Espresso example](4-2-quantum-espresso.ipynb) only sets the launch command of the calculator.

In [ ]:
def run_pi_simulation(n_intervals, executable, mpi_launcher, mpi_ranks):
    # Executed in the working directory of the SLURM job: write the input, run the MPI code, parse the output.
    import subprocess

    with open("input.txt", "w") as f:
        f.write(f"n_intervals {n_intervals}\n")
    with open("simulation.log", "w") as log:
        subprocess.run(
            mpi_launcher + ["-n", str(mpi_ranks), executable],
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
        )
    with open("output.txt") as f:
        output = dict(line.split() for line in f)
    return {"n_intervals": n_intervals, "pi": float(output["pi"]), "mpi_ranks": int(output["mpi_ranks"])}

## Step 3: Request the resources
The resources of the simulation are defined in the `resource_dict` - with one important difference to MPI-parallel
Python functions:

* `cores` is set to **one**. `cores` is the number of MPI ranks executorlib uses to start the *Python function*. With
  `cores=2` executorlib would start the function with `srun -n 2`, so two copies of the wrapper would each start the
  simulation code.
* `threads_per_core` reserves the CPU cores for the executable. The `SlurmClusterExecutor` multiplies `cores` and
  `threads_per_core` before it renders the submission template, so `{{cores}}` - and with it `#SBATCH --ntasks` -
  equals `cores_per_simulation`. The `srun` call inside the job then uses these tasks for the MPI ranks of the
  executable.
* `cwd` sets the working directory of the job: every simulation gets its own directory for its input, output and log
  files, while the cache stays in the `cache_directory`.

In [ ]:
def get_resources(n_intervals):
    return {
        "cores": 1,  # the Python wrapper itself is serial - it must not be started as MPI program
        "threads_per_core": cores_per_simulation,  # CPU cores for the MPI ranks of the executable
        "cwd": os.path.abspath(os.path.join("story11_runs", f"n_intervals_{n_intervals}")),
        "run_time_max": 300,
    }

## Step 4: Run the convergence study
The workflow logic stays in Python: one simulation per number of intervals, and a serial analysis step which compares
the results to the exact value. The futures of the simulations are passed to the analysis as arguments, so it is
submitted right away as a SLURM job which depends on the simulation jobs ([Story 2](story-2-multi-resource-pipeline.ipynb)).

In [ ]:
from executorlib import SlurmClusterExecutor

cache_directory = os.path.abspath("story11_cache")
n_intervals_lst = [10, 100, 1000]


def analyse_convergence(*result_lst):
    import math

    return {result["n_intervals"]: abs(result["pi"] - math.pi) for result in result_lst}


with SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template},
) as exe:
    simulation_lst = [
        exe.submit(
            run_pi_simulation,
            n_intervals,
            executable=executable,
            mpi_launcher=mpi_launcher,
            mpi_ranks=cores_per_simulation,
            resource_dict=get_resources(n_intervals),
        )
        for n_intervals in n_intervals_lst
    ]
    error = exe.submit(analyse_convergence, *simulation_lst, resource_dict={"cores": 1, "run_time_max": 60})
    result_lst = [simulation.result() for simulation in simulation_lst]
    print(error.result())

result_lst

## Step 5: Inspect a working directory
The raw files of every simulation remain in its working directory - next to `time.out`, the output of the SLURM job -
so they can be inspected, archived or parsed again with a different parser later.

In [ ]:
run_directory = get_resources(1000)["cwd"]
print(sorted(os.listdir(run_directory)))
with open(os.path.join(run_directory, "output.txt")) as f:
    print(f.read())

Executing the cell of Step 4 again reloads all results from the cache without submitting a job. Keep in mind that the
cache key is a hash of the Python function and its arguments - the path of the executable, not its content. After
recompiling or updating the simulation code, use a new `cache_directory` or pass the version of the code as an
additional argument.

## Other codes and other launchers
The pattern is the same for every MPI-parallel executable: Quantum Espresso (`pw.x`), LAMMPS (`lmp -in in.lammps`),
VASP, CP2K or GROMACS - only the input writer and the output parser change. The
[Quantum Espresso example](4-2-quantum-espresso.ipynb) is a complete real-world version of this story: it computes an
energy-volume curve with ASE, where the calculator starts `pw.x` with `flux run`, and uses the same `resource_dict`
with `cores=1`, `threads_per_core` and `cwd`. If your code also provides Python bindings which accept an mpi4py
communicator - like LAMMPS or [GPAW](4-1-gpaw.ipynb) - you can alternatively call it directly in an MPI-parallel
Python function, see [Story 12](story-12-mpi4py-functions.ipynb).

Inside an existing allocation, the `SlurmJobExecutor` starts every function as an `srun` job step, and starting the
executable with another `srun` from within this job step is not reliably supported by SLURM. Start the
[flux](https://flux-framework.org) scheduler in the allocation instead (`srun flux start python my_workflow.py`), use
the `FluxJobExecutor` with the same `resource_dict` and `mpi_launcher = ["flux", "run"]`, as shown in the
[Quantum Espresso example](4-2-quantum-espresso.ipynb) and in
[SLURM with flux](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html#slurm-with-flux).

## Clean up

In [ ]:
import shutil

for directory in [cache_directory, os.path.abspath("story11_runs"), code_directory]:
    shutil.rmtree(directory, ignore_errors=True)

## What you learned
* An MPI-parallel executable is coupled to Python with a serial wrapper function, which writes the input, starts the
  executable with the MPI launcher via `subprocess` and parses the output.
* Such wrappers are submitted with `cores=1`; the CPU cores for the executable are requested with `threads_per_core`,
  which the `SlurmClusterExecutor` multiplies with `cores` for `{{cores}}` in the submission template.
* `cwd` gives every simulation its own working directory, which keeps the raw files for inspection.
* Futures of the simulations connect them to the analysis in Python, just like any other function.

**Reference:** [Quantum Espresso example](4-2-quantum-espresso.ipynb),
[HPC Cluster Executor](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html),
[cores, threads per core and maximum workers](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#cores-threads-per-core-and-maximum-workers)
and the [resource dictionary](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary).